## Analisis Sentimen Aplikasi dan Gerai Kopi Kenangan
### Data Preprocessing (Functional Approach)

In [1]:
pip install Sastrawi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import string
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

### Kamus Slang (Slang Dictionary)

In [3]:
import json

slang_path = 'combined_slang_words.txt'
if not os.path.exists(slang_path):
    slang_path = os.path.join('preprocessing', 'combined_slang_words.txt')

with open(slang_path, 'r', encoding='utf-8') as f:
    slang_dict = json.load(f)

print(f"Kamus slang didefinisikan dengan {len(slang_dict)} kata.")

Kamus slang didefinisikan dengan 1018 kata.


### Fungsi Pembersih Teks & Preprocessing

In [4]:
stopwords_path = 'combined_stop_words.txt'
if not os.path.exists(stopwords_path):
    stopwords_path = os.path.join('preprocessing', 'combined_stop_words.txt')

with open(stopwords_path, 'r', encoding='utf-8') as f:
    stopword_indo = set([line.strip() for line in f if line.strip()])

print(f"Stopwords didefinisikan dengan {len(stopword_indo)} kata.")

print("Membuat Sastrawi Stemmer...")
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stem_cache = {}

Stopwords didefinisikan dengan 673 kata.
Membuat Sastrawi Stemmer...


In [5]:
def get_cached_stem(word, stemmer_obj):
    if not word:
        return ""
    if word not in stem_cache:
        stem_cache[word] = stemmer_obj.stem(word)
    return stem_cache[word]

In [6]:
def clean_noise(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'\.{2,}\s*Lainnya', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.replace('\n', ' ')
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    return text.lower()

In [7]:
def preprocess_review(text):
    try:
        cleaned = clean_noise(text)
        if not cleaned:
            return ""
        tokens = word_tokenize(cleaned)
        normalized = [slang_dict.get(t, t) for t in tokens]
        filtered = [t for t in normalized if t not in stopword_indo]
        stemmed = [get_cached_stem(t, stemmer) for t in filtered if t]
        return ' '.join(stemmed)
    except Exception as e:
        print(f"[ERROR] Failed to process review: '{text[:100]}...' - Error: {e}")
        return ""

### Load Dataset & Jalankan Pipeline

In [8]:
dataset_path = 'cleaned_data.csv'
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset berhasil dimuat! Jumlah data: {len(df)} baris.")
    print(df.head(2))
else:
    print(f"[ERROR] File {dataset_path} tidak ditemukan di folder root!")

Dataset berhasil dimuat! Jumlah data: 2938 baris.
     nama_pengulas                                             ulasan  \
0            Immia  Sekarang udah jadi Kenangan boulangerie, semak...   
1  Anton Kristanto  Outlet kopi kenangan versi premium... The best...   

      rating                       nama_gerai  
0  5 bintang  Kopi Kenangan - Kota Kasablanka  
1  5 bintang  Kopi Kenangan - Kota Kasablanka  


In [9]:
print("Menjalankan text preprocessing pada seluruh ulasan...")
df['text_clean'] = df['ulasan'].apply(preprocess_review)
print("Pembersihan selesai!")

Menjalankan text preprocessing pada seluruh ulasan...
Pembersihan selesai!


In [10]:
empty_cleaned_reviews = df[df['text_clean'] == ''].shape[0]
total_reviews = df.shape[0]
print(f"Jumlah ulasan yang menjadi kosong setelah preprocessing: {empty_cleaned_reviews} dari {total_reviews}")
if empty_cleaned_reviews == total_reviews:
    print("Semua ulasan menjadi kosong setelah preprocessing. Ini adalah masalah utama.")

Jumlah ulasan yang menjadi kosong setelah preprocessing: 12 dari 2938


### Inspecting Preprocessing Results

In [11]:
print("Original vs. Preprocessed Reviews (First 10 samples):")
for i in range(10):
    original_review = df['ulasan'].iloc[i]
    preprocessed_review = df['text_clean'].iloc[i]
    print(f"\nReview #{i+1}:")
    print(f"  Original: {original_review}")
    print(f"  Cleaned:  {preprocessed_review}")

Original vs. Preprocessed Reviews (First 10 samples):

Review #1:
  Original: Sekarang udah jadi Kenangan boulangerie, semakin keren tempatnya dan lebih banyak pilihan rotinya. Ada mesin untuk self order juga. Ada banyak colokan juga buat yang mau sambil wfc. Kopi nya juga makin banyak pilihan walau kopi kenangan mantan masih jadi favorit 😄 … Lainnya
  Cleaned:  kenang boulangerie keren tempat pilih roti mesin self order colok wfc kopi nya pilih kopi kenang favorit

Review #2:
  Original: Outlet kopi kenangan versi premium... The best banget pelayanan dan suasananya. Sangat mewah
Jenis pe…
Lainnya
  Cleaned:  outlet kopi kenang versi premium best banget layan suasana mewah pe

Review #3:
  Original: kalo makan siang sering beli kopken walaupun antrian panjang banget tapi pelayanannya tetep gercep salut banget👍kopinya juga enakkk banget😋🫰 … Lainnya
  Cleaned:  makan siang beli kopken antri banget layan tetep gercep salut bangetkopinya enakkk banget

Review #4:
  Original: Semuanya pasti

### Pelabelan Sentimen berdasarkan Rating & Filtering

In [12]:
df['word_count'] = df['text_clean'].apply(lambda x: len(str(x).split()))

In [13]:
print(f"Jumlah baris sebelum difilter: {len(df)}")
df_filtered = df[df['word_count'] >= 1].copy()
print(f"Jumlah baris setelah difilter (minimal 1 kata): {len(df_filtered)}")

Jumlah baris sebelum difilter: 2938
Jumlah baris setelah difilter (minimal 1 kata): 2926


In [14]:
def map_rating_to_sentiment(rating_str):
    if not isinstance(rating_str, str):
        return 'neutral'
    match = re.search(r'\d', rating_str)
    if match:
        stars = int(match.group())
        if stars >= 4:
            return 'positive'
        elif stars <= 2:
            return 'negative'
    return 'neutral'

In [15]:
if 'rating' in df_filtered.columns:
    df_filtered['sentiment'] = df_filtered['rating'].apply(map_rating_to_sentiment)
    print("\nDistribusi Kelas Sentimen:")
    print(df_filtered['sentiment'].value_counts())
else:
    print("[WARNING] Kolom rating tidak ditemukan, ulasan tidak dapat dilabeli otomatis.")


Distribusi Kelas Sentimen:
sentiment
positive    2300
negative     498
neutral      128
Name: count, dtype: int64


### Hasil Preprocessing

In [16]:
output_path = 'cleaned_reviews.csv'
df_filtered.to_csv(output_path, index=False)
print(f"Proses preprocessing berhasil diselesaikan!")
print(f"Data bersih disimpan ke: {output_path}")

Proses preprocessing berhasil diselesaikan!
Data bersih disimpan ke: cleaned_reviews.csv
